In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/upwork_jobs.csv")
print(df.shape)
df.head()

(53058, 9)


,title,link,description,published_date,is_hourly,hourly_low,hourly_high,budget,country
0,Experienced Media Buyer For Solar Pannel and R...,https://www.upwork.com/jobs/Experienced-Media-...,We’re looking for a talented and hardworking a...,2024-02-17 09:09:54+00:00,False,NaN,NaN,500.0,NaN
1,Full Stack Developer,https://www.upwork.com/jobs/Full-Stack-Develop...,Job Title: Full Stack DeveloperWe are seeking ...,2024-02-17 09:09:17+00:00,False,NaN,NaN,1100.0,United States
2,SMMA Bubble App,https://www.upwork.com/jobs/SMMA-Bubble-App_%7...,I need someone to redesign my bubble.io site t...,2024-02-17 09:08:46+00:00,True,10.0,30.0,NaN,United States
3,Talent Hunter Specialized in Marketing,https://www.upwork.com/jobs/Talent-Hunter-Spec...,Join Our Growing Team!We are an innovative com...,2024-02-17 09:08:08+00:00,NaN,NaN,NaN,NaN,United States
4,Data Engineer,https://www.upwork.com/jobs/Data-Engineer_%7E0...,We are looking for a resource who can work par...,2024-02-17 09:07:42+00:00,False,NaN,NaN,650.0,India


In [2]:

df = df.dropna(subset=["description"])

def resolve_price(row):
    if row["is_hourly"] == True or row["is_hourly"] == "True":
        low, high = row.get("hourly_low"), row.get("hourly_high")
        if pd.notna(low) and pd.notna(high):
            avg_rate = (low + high) / 2
            return avg_rate * 40  # assume ~40 hour project
        return np.nan
    else:
        return row.get("budget")

df["resolved_price"] = df.apply(resolve_price, axis=1)

df = df.dropna(subset=["resolved_price"])
df = df[(df["resolved_price"] > 20) & (df["resolved_price"] < 20000)]  # strip outliers/junk rows

print(df.shape)
df[["title", "description", "resolved_price"]].head()

(39262, 10)


,title,description,resolved_price
0,Experienced Media Buyer For Solar Pannel and R...,We’re looking for a talented and hardworking a...,500.0
1,Full Stack Developer,Job Title: Full Stack DeveloperWe are seeking ...,1100.0
2,SMMA Bubble App,I need someone to redesign my bubble.io site t...,800.0
4,Data Engineer,We are looking for a resource who can work par...,650.0
7,need Portuguese writers who can understand and...,Portuguese writers who have a deep understandi...,580.0


In [7]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sample_df = df.sample(n=min(10000, len(df)), random_state=42).reset_index(drop=True)

sample_df["text_for_match"] = sample_df["title"].astype(str) + " " + sample_df["description"].astype(str)

tfidf = TfidfVectorizer(max_features=8000, stop_words="english", ngram_range=(1, 2), min_df=2)
tfidf_matrix = tfidf.fit_transform(sample_df["text_for_match"])

print(tfidf_matrix.shape)

(10000, 8000)


In [8]:
def estimate_base_price(new_description: str, k: int = 15, min_similarity: float = 0.05) -> dict:
    query_vec = tfidf.transform([new_description])
    sims = cosine_similarity(query_vec, tfidf_matrix)[0]

    top_k_idx = np.argsort(sims)[-k:][::-1]
    top_matches = sample_df.iloc[top_k_idx].copy()
    top_matches["similarity"] = sims[top_k_idx]

    top_matches = top_matches[top_matches["similarity"] >= min_similarity]

    if len(top_matches) == 0:
        
        return {
            "estimated_base_price": float(np.median(sample_df["resolved_price"])),
            "top_matches": [],
            "fallback_used": True,
        }

    weights = top_matches["similarity"].values
    prices = top_matches["resolved_price"].values
    estimated_price = float(np.average(prices, weights=weights))

    return {
        "estimated_base_price": round(estimated_price, 2),
        "top_matches": top_matches[["title", "resolved_price", "similarity"]].to_dict("records"),
        "fallback_used": False,
    }

In [9]:
result = estimate_base_price("Build a React dashboard for an enterprise client, scalable architecture needed")
print("Estimated base price:", result["estimated_base_price"])
for m in result["top_matches"][:5]:
    print(f"  ${m['resolved_price']:.0f}  sim={m['similarity']:.3f}  {m['title']}")

Estimated base price: 1555.72
  $3800  sim=0.209  Heap Analytics Dashboard Development
  $900  sim=0.200  ReactNative Build Support
  $520  sim=0.197  Work on react native project
  $1500  sim=0.195  Javascript Dev Needed - React / Firebase - Experienced Devs Only! Short term intensive position
  $860  sim=0.194  React developer for B2B automation platform


In [10]:
result2 = estimate_base_price("Simple quick landing page fix, urgent, small task")
print("Estimated base price:", result2["estimated_base_price"])
for m in result2["top_matches"][:5]:
    print(f"  ${m['resolved_price']:.0f}  sim={m['similarity']:.3f}  {m['title']}")

Estimated base price: 764.95
  $75  sim=0.443  Creating Bridge Pages using OptimizePress 3.0
  $2800  sim=0.408  Build a clean, WordPress landing page for my product that will be launching soon
  $1200  sim=0.393  Writer to create landing page and sales page for new course
  $1200  sim=0.381  Build a highly optimized landing page for my personal injury law firm using Wordpress
  $600  sim=0.367  Looking For Figma Designer For A Landing Page Changes
